In [1]:
from transformers import AutoTokenizer

# Load mBERT tokenizer (cased, multilingual)
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
print("mBERT tokenizer loaded successfully.")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

mBERT tokenizer loaded successfully.


In [2]:
import pandas as pd

# Use the paths you provided earlier for the mBERT dataset
small = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mbert-dataset/small.csv")
medium = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mbert-dataset/medium.csv")
large = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mbert-dataset/large.csv")

# Add size category for reference
small['size_category'] = 'small'
medium['size_category'] = 'medium'
large['size_category'] = 'large'

# Combine all into one DataFrame
all_data = pd.concat([small, medium, large], ignore_index=True)
print("Total rows:", all_data.shape[0])
print("Columns:", all_data.columns.tolist())
print(all_data.head(2))

Total rows: 50529
Columns: ['id', 'content', 'summary', 'size_category']
    id                                            content  \
0  185  लक्ष्मी इन्टरकन्टिनेन्टल प्रा. लि.ले हुन्डाई इ...   
1  189  बेइजिङ। चीनले अमेरिका, युरोपेली संघ (इयु), ताइ...   

                                             summary size_category  
0  लक्ष्मी इन्टरकन्टिनेन्टलले हुन्डाईका इलेक्ट्रि...         small  
1  चीनले अमेरिका, युरोपेली संघ, ताइवान र जापानबाट...         small  


In [3]:
from sklearn.model_selection import train_test_split

# 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train size:", train_df.shape[0])
print("Validation size:", val_df.shape[0])
print("Test size:", test_df.shape[0])

Train size: 40423
Validation size: 5053
Test size: 5053


In [4]:
from datasets import Dataset

# Convert pandas DataFrames to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[['content', 'summary']])
val_dataset   = Dataset.from_pandas(val_df[['content', 'summary']])
test_dataset  = Dataset.from_pandas(test_df[['content', 'summary']])

MAX_INPUT_LENGTH = 512   # mBERT max
MAX_TARGET_LENGTH = 128  # summaries

def tokenize_function(examples):
    # Tokenize the source articles
    model_inputs = tokenizer(
        examples['content'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    # Tokenize the target summaries (no as_target_tokenizer needed)
    labels = tokenizer(
        examples['summary'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("Tokenizing training set...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing validation set...")
tokenized_val   = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing test set...")
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

# Remove raw text columns (already tokenized)
tokenized_train = tokenized_train.remove_columns(['content', 'summary'])
tokenized_val   = tokenized_val.remove_columns(['content', 'summary'])
tokenized_test  = tokenized_test.remove_columns(['content', 'summary'])

print("Tokenization complete.")
print("Train features:", tokenized_train.features)
print("Validation features:", tokenized_val.features)
print("Test features:", tokenized_test.features)

Tokenizing training set...


Map:   0%|          | 0/40423 [00:00<?, ? examples/s]

Tokenizing validation set...


Map:   0%|          | 0/5053 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/5053 [00:00<?, ? examples/s]

Tokenization complete.
Train features: {'__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
Validation features: {'__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
Test features: {'__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [5]:
from transformers import EncoderDecoderModel

# Load mBERT as both encoder and decoder (shared architecture)
encoder_decoder = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "bert-base-multilingual-cased",  # encoder
    "bert-base-multilingual-cased"   # decoder
)

# Set special tokens for generation
encoder_decoder.config.decoder_start_token_id = tokenizer.cls_token_id  # [CLS]
encoder_decoder.config.eos_token_id = tokenizer.sep_token_id            # [SEP]
encoder_decoder.config.pad_token_id = tokenizer.pad_token_id            # [PAD]

# The model has ~335M parameters
print("EncoderDecoder model created.")
print("Total parameters:", encoder_decoder.num_parameters())


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertLMHeadModel LOAD REPORT from: bert-base-multilingual-cased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
cls.seq_relationship.weight                                        | UNEXPECTED | 
bert.pooler.dense.bias                                             | UNEXPECTED | 
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.bias         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.bias         | MISSING    | 
bert.encoder.layer.{0...

EncoderDecoder model created.
Total parameters: 384194811


In [6]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Data collator – dynamically pads each batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=encoder_decoder,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mbert-nepali-summarizer",
    eval_strategy="epoch",                   # evaluate each epoch
    save_strategy="epoch",                   # save each epoch
    logging_strategy="steps",
    logging_steps=500,
    per_device_train_batch_size=8,           # mBERT is smaller than IndicBART
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,           # effective batch size = 16
    num_train_epochs=3,
    learning_rate=3e-5,                      # slightly lower for encoder‑decoder
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,                               # mixed precision
    predict_with_generate=True,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

print("Training arguments ready.")
print("Output dir:", training_args.output_dir)
print("Batch size per GPU:", training_args.per_device_train_batch_size)
print("Effective batch size:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

Training arguments ready.


Output dir: ./mbert-nepali-summarizer
Batch size per GPU: 8
Effective batch size: 16


In [7]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=encoder_decoder,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

# Assign the tokenizer for generation during evaluation
trainer.tokenizer = tokenizer

print("Starting training...")
trainer.train()
print("Training finished.")

Starting training...


/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:445: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
from transformers import EncoderDecoderModel, AutoTokenizer

# Path to the best checkpoint (last epoch)
checkpoint_path = "./mbert-nepali-summarizer/checkpoint-3792"

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = EncoderDecoderModel.from_pretrained(checkpoint_path)

print("Model and tokenizer loaded from checkpoint.")

In [ ]:
model.save_pretrained("./mbert-final")
tokenizer.save_pretrained("./mbert-final")
print("Final model and tokenizer saved to ./mbert-final")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/mbert-final", 'zip', "/kaggle/working/mbert-final")
print("ZIP created: /kaggle/working/mbert-final.zip")